# Experiment 20 — Theme Business Needs + Stage — Theme Batch by Stage ID

## What the LLM sees
Shared Theme context + unique Stage data + Stage-specific candidate L3s. No source record key, no artificial item_id, no ground truth.

In [ ]:
from pathlib import Path
from time import perf_counter
import ast, json, os
import pandas as pd
from IPython.display import display
from common import call_llm_with_metrics, load_gateway, parse_json_response, save_results_excel, score_sets

NOTEBOOK_DIR = Path.cwd()

def resolve_data_path(relative_path, env_var):
    override = os.getenv(env_var)
    if override:
        return Path(override).expanduser()
    rel = Path(relative_path)
    for root in [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR / "l3_experiments"]:
        p = root / rel
        if p.exists():
            return p
    raise FileNotFoundError(relative_path)

PARQUET_PATH = resolve_data_path("full_golden.parquet", "L3_FULL_GOLDEN_PATH")
STAGE_PATH = resolve_data_path("VSSrv.csv", "L3_STAGE_PATH")
STAGE_CAPABILITY_MAP_PATH = resolve_data_path("VSSCaprv (1).csv", "L3_STAGE_CAPABILITY_MAP_PATH")
GROUND_TRUTH_PATH = resolve_data_path("results/epic_l3_ground_truth_full_golden.xlsx", "L3_GROUND_TRUTH_PATH")
SAMPLE_SIZE, SAMPLE_SEED = 50, 42

def clean_text(v):
    if v is None:
        return ""
    try:
        if pd.isna(v): return ""
    except (TypeError, ValueError):
        pass
    return str(v).strip()

def parse_list(v):
    if v is None: return []
    if isinstance(v, (list, tuple, set)): return [clean_text(x) for x in v if clean_text(x)]
    try:
        if pd.isna(v): return []
    except (TypeError, ValueError):
        pass
    s = str(v).strip()
    if not s: return []
    try: x = ast.literal_eval(s)
    except (SyntaxError, ValueError): return [s]
    if isinstance(x, (list, tuple, set)): return [clean_text(i) for i in x if clean_text(i)]
    return [clean_text(x)] if clean_text(x) else []

def read_table(path, sheet_name=None):
    path = Path(path)
    if path.suffix.lower() == ".parquet": return pd.read_parquet(path)
    if path.suffix.lower() == ".csv": return pd.read_csv(path, dtype=str, encoding="cp1252", encoding_errors="replace")
    return pd.read_excel(path, sheet_name=sheet_name, dtype=str)

population = read_table(GROUND_TRUTH_PATH, "evaluation_population")
population = population.drop_duplicates(["theme_key", "epic_key"]).sort_values(["theme_key", "epic_key"]).reset_index(drop=True)
if len(population) < SAMPLE_SIZE:
    raise ValueError(f"Need {SAMPLE_SIZE} valid records, found {len(population)}")
evaluation_population = population.sample(SAMPLE_SIZE, random_state=SAMPLE_SEED).sort_values(["theme_key", "epic_key"]).reset_index(drop=True)
selected_pairs = set(zip(evaluation_population.theme_key, evaluation_population.epic_key))

frame = read_table(PARQUET_PATH)
themes = {}
for _, row in frame.iterrows():
    theme_id = clean_text(row.get("key"))
    selected = [k for k in parse_list(row.get("epic_keys")) if (theme_id, k) in selected_pairs]
    if selected:
        themes[theme_id] = {
            "theme_description": clean_text(row.get("description")),
            "theme_business_needs": clean_text(row.get("businessNeeds")),
        }

stage_frame = read_table(STAGE_PATH)
stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)

def stage_context(stage_id):
    m = stage_frame.loc[stage_frame["Value Stream Stage ID"].astype(str).str.strip().eq(stage_id)]
    if m.empty: raise KeyError(f"No stage metadata for {stage_id}")
    r = m.iloc[0]
    return {
        "stage_id": stage_id,
        "stage_name": clean_text(r["Value Stream Stage Name"]),
        "stage_description": clean_text(r["Value Stream Stage Description"]),
        "entrance_criteria": clean_text(r["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean_text(r["Value Stream Stage Exit Criteria"]),
    }

def candidates_for_stage(stage_id):
    rows = stage_capability_map.loc[stage_capability_map["Value Stream Stage ID"].astype(str).str.strip().eq(stage_id)].copy()
    rows = rows.drop_duplicates("Capability ID").sort_values(["Capability Name", "Capability ID"])
    return [{
        "capability_id": clean_text(r["Capability ID"]),
        "capability_name": clean_text(r["Capability Name"]),
        "capability_description": clean_text(r["Capability Description"]),
        "capability_tier": clean_text(r["Capability Tier"]),
    } for _, r in rows.iterrows()]

def build_batch_stages(theme_rows):
    stage_ids = sorted({sid for raw in theme_rows["stage_ids"] for sid in parse_list(raw)})
    stages, allowed = [], {}
    for sid in stage_ids:
        cands = candidates_for_stage(sid)
        stages.append({**stage_context(sid), "candidate_l3_capabilities": cands})
        allowed[sid] = [c["capability_id"] for c in cands]
    return stages, allowed

print(f"Selected {len(evaluation_population)} valid records across {evaluation_population['theme_key'].nunique()} Themes.")


## Production prompt

In [ ]:
EXPERIMENT_NAME = "E20_BUSINESS_NEEDS_STAGE_BATCH_BY_STAGE_ID"

SYSTEM_PROMPT = 'You are performing Level 3 business capability classification for multiple Value Stream Stages that share the same Theme Business Needs.\n\nAn L3 capability is a Level 3 business capability: a specific business function within the enterprise capability hierarchy.\n\nTheme Business Neds is the shared business evidence.\n\nClassify each supplied Value Stream Stage independently using only:\n- the shared Theme Business Needs,\n- that Stage\'s data,\n- that Stage\'s candidate L3 capabilities.\n\nEVIDENCE\n\nTheme Business Needs describes the shared business outcomes, requirements, and functions that need to be delivered.\nEach Stage identifies the portion of those Business Needs relevant to its own classification.\nFor each candidate L3, capability_description is the primary semantic definition; capability_name is a supporting label; capability_tier is taxonomy context only; capability_id is only the exact identifier to return.\nDo not infer business meaning from capability_id.\n\nCLASSIFICATION\n\nFor each Stage independently:\n1. Identify the business functions required by the shared Theme Business Needs.\n2. Use that Stage\'s data to focus on the relevant portion of those needs.\n3. Compare the evidence only against that Stage\'s candidate L3 capability descriptions.\n4. Select every candidate whose business function is directly supported.\n\nDo not use one Stage\'s data or candidates as evidence for another Stage.\nTo not select a capability merely because it belongs to the Stage, shares terminology, is broadly related, is upstream/downstream, or commonly enables another capability.\nOnly return capability_id values supplied for that Stage.\nIf none are supported, return an empty list.\nReturn exactly one result for every supplied stage_id.\n\nOUTPUT:\nReturn JSON only:\n{"stages":[{"stage_id":"VSS000123","l3":["CAP00000123"]},{"stage_id":"VSS000456","l3":[]}]}\nDo not return reasons, explanations, Markdown, or additional fields.'

def build_user_prompt(theme, stages):
    payload = {"theme_business_needs": theme["theme_business_needs"], "stages": stages}
    return json.dumps(payload, ensure_ascii=False, indent=2)


## Prompt preview — printed exactly as sent

In [ ]:
preview_theme_id, preview_rows = next(iter(evaluation_population.groupby("theme_key", sort=True)))
preview_stages, _ = build_batch_stages(preview_rows)
preview_user_prompt = build_user_prompt(themes[preview_theme_id], preview_stages)
print("SYSTEM PROMPT")
print("="*80)
print(SYSTEM_PROMPT)
print("\nFORMATTED USER PROMPT")
print("="*80)
print(preview_user_prompt)


## Prediction and evaluation

In [ ]:
def validate_batch(payload, expected_stage_ids, allowed_by_stage):
    if not isinstance(payload, dict) or set(payload) != {"stages"} or not isinstance(payload["stages"], list):
        raise ValueError("Response must contain only a stages list")
    expected = set(expected_stage_ids)
    out = {}
    for item in payload["stages"]:
        if not isinstance(item, dict) or set(item) != {"stage_id", "l3"}:
            raise ValueError("Each result must contain exactly stage_id and l3")
        sid = str(item["stage_id"]).strip()
        if sid not in expected or sid in out or not isinstance(item["l3"], list):
            raise ValueError(f"Invalid stage result: {sid}")
        allowed, seen, selected = set(allowed_by_stage[sid]), set(), []
        for cid in item["l3"]:
            if not isinstance(cid, str) or cid.strip() not in allowed or cid.strip() in seen:
                raise ValueError(f"Invalid L3 selection for {sid}: {cid}")
            cid = cid.strip(); seen.add(cid); selected.append(cid)
        out[sid] = selected
    if set(out) != expected:
        raise ValueError(f"Missing stage results: {sorted(expected-set(out))}")
    return out

def predict_theme(gateway, theme_id, theme_rows):
    stages, allowed = build_batch_stages(theme_rows)
    user_prompt = build_user_prompt(themes[theme_id], stages)
    raw, metrics = call_llm_with_metrics(gateway, SYSTEM_PROMPT, user_prompt, id="9zdn8n", reasoning_effort="low")
    pred = validate_batch(parse_json_response(raw), [s["stage_id"] for s in stages], allowed)
    return pred, metrics

def run_experiment():
    gateway = load_gateway(); results=[]; calls=[]
    for theme_id, theme_rows in evaluation_population.groupby("theme_key", sort=True):
        started = perf_counter()
        try:
            stage_pred, metrics = predict_theme(gateway, theme_id, theme_rows)
            calls.append({"experiment":EXPERIMENT_NAME,"theme_id":theme_id,"record_count":len(theme_rows),"stage_count":len(stage_pred),"status":"ok",**metrics,"error":None})
            for _, row in theme_rows.iterrows():
                stage_ids = parse_list(row["stage_ids"])
                pred = sorted({cid for sid in stage_ids for cid in stage_pred.get(sid, [])})
                truth = parse_list(row["gt_l3_ids"])
                results.append({"experiment":EXPERIMENT_NAME,"theme_id":theme_id,"record_key":clean_text(row["epic_key"]),"stage_ids":stage_ids,"predicted_l3_ids":pred,"gt_l3_ids":truth,"status":"ok","error":None,**score_sets(pred,truth)})
        except Exception as exc:
            calls.append({"experiment":EXPERIMENT_NAME,"theme_id":theme_id,"record_count":len(theme_rows),"stage_count":None,"status":"error","latency_seconds":perf_counter()-started,"input_tokens":None,"output_tokens":None,"total_tokens":None,"error":str(exc)})
            for _, row in theme_rows.iterrows():
                truth=parse_list(row["gt_l3_ids"])
                results.append({"experiment":EXPERIMENT_NAME,"theme_id":theme_id,"record_key":clean_text(row["epic_key"]),"stage_ids":parse_list(row["stage_ids"]),"predicted_l3_ids":None,"gt_l3_ids":truth,"status":"error","error":str(exc),"exact_match":None,"precision":None,"recall":None,"f1":None,"predicted_count":None,"truth_count":len(truth)})
    return pd.DataFrame(results), pd.DataFrame(calls)

results, call_metrics = run_experiment()
scored = results[results.status.eq("ok")]
ok_calls = call_metrics[call_metrics.status.eq("ok")]
summary = pd.DataFrame([{"evaluated_records":len(scored),"exact_match_accuracy":scored.exact_match.mean() if len(scored) else 0,"mean_precision":scored.precision.mean() if len(scored) else 0,"mean_recall":scored.recall.mean() if len(scored) else 0,"mean_f1":scored.f1.mean() if len(scored) else 0}])
latency_tokens = pd.DataFrame([{"successful_calls":len(ok_calls),"failed_calls":int(call_metrics.status.eq("error").sum()),"avg_latency_seconds":ok_calls.latency_seconds.mean() if len(ok_calls) else None,"p50_latency_seconds":ok_calls.latency_seconds.quantile(.5) if len(ok_calls) else None,"p95_latency_seconds":ok_calls.latency_seconds.quantile(.95) if len(ok_calls) else None,"total_input_tokens":ok_calls.input_tokens.sum() if len(ok_calls) else 0,"total_output_tokens":ok_calls.output_tokens.sum() if len(ok_calls) else 0,"total_tokens":ok_calls.total_tokens.sum() if len(ok_calls) else 0,"tokens_per_scored_record":ok_calls.total_tokens.sum()/len(scored) if len(scored) else None}])
display(summary); display(latency_tokens); display(call_metrics); display(results.head(50))
output_path = save_results_excel(results, EXPERIMENT_NAME, "results", extra_sheets={"evaluation_summary":summary,"llm_metrics":call_metrics,"latency_tokens":latency_tokens,"evaluation_population":evaluation_population})
print(f"Saved {output_path}")
